In [ ]:
import seaborn as sns
import pandas as pd
import os

df = sns.load_dataset("titanic")
print(df.shape)             
print(df.columns.tolist())   

df["age"] = df["age"].fillna(df.groupby(["pclass", "sex"])["age"].transform("median"))
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns="deck", errors="ignore")

print(df.isna().sum()[lambda s: s > 0])


(891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
embarked    2
dtype: int64


In [ ]:
print("Lignes en double :", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6)

Lignes en double : 118


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
263,0,1,male,40.0,0,0,0.0000,S,First,man,True,Southampton,no,True
633,0,1,male,40.0,0,0,0.0000,S,First,man,True,Southampton,no,True
815,0,1,male,40.0,0,0,0.0000,S,First,man,True,Southampton,no,True
30,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True
64,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True
295,0,1,male,40.0,0,0,27.7208,C,First,man,True,Cherbourg,no,True


In [ ]:
df = df.drop(columns=["alive", "class", "embarked", "who", "adult_male", "alone"])

df["pclass"] = pd.Categorical(df["pclass"], categories=[1, 2, 3], ordered=True)
df["sex"] = df["sex"].astype("category")
df["embark_town"] = df["embark_town"].astype("category")

df.dtypes

survived          int64
pclass         category
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
embark_town    category
dtype: object

In [21]:
print("Billets à 0 :", (df["fare"] == 0).sum())
print("Billets > 500 :", (df["fare"] > 500).sum())

def seuil(s):
    return s.quantile(0.75) + 1.5 * (s.quantile(0.75) - s.quantile(0.25))

seuils = df.groupby("pclass", observed=True)["fare"].transform(seuil)
hors_bornes = df[df["fare"] > seuils]
print(hors_bornes.groupby("pclass", observed=True).size())

Billets à 0 : 15
Billets > 500 : 3
pclass
1    20
2     7
3    52
dtype: int64


In [ ]:
# 5. Enregistrement
import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/titanic_propre.csv", index=False)
print(pd.read_csv("data/titanic_propre.csv").shape)   # attendu : (891, 8)